# NFL 4th Down — Aggressive Coaches Analysis
Is Campbell's aggression unique, or is there a new breed of analytically-minded
aggressive coaches? Does aggression itself predict good decisions, or is it neutral?

**Central questions:**
- Who are the most aggressive coaches in the data?
- Does going for it more often correlate with better or worse decision quality?
- How does Campbell compare to other aggressive coaches over time?

**Inputs:** `data/fourth_downs_graded.parquet`, `data/raw/games_coaches.parquet`, `outputs/coach_grades.csv`  
**Outputs:** three charts in `outputs/figures/`, `outputs/aggressive_coaches.csv`

In [ ]:
import sys
sys.path.append('../src')

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

DATA_DIR = '../data/'
SAVE_DIR = '../outputs/figures/'
OUT_DIR  = '../outputs/'
os.makedirs(SAVE_DIR, exist_ok=True)

CAMPBELL_COLOR = '#C5203B'
NEUTRAL_COLOR  = '#546E7A'
GO_COLOR       = '#2E7D32'

# Comparison coaches for the season-trend chart
# Chosen to span archetypes and give good narrative contrast with Campbell
COMPARISON_COACHES = [
    'Dan Campbell',
    'Andy Reid',
    'Kyle Shanahan',
    'Bill Belichick',
    'Mike McCarthy',
]
COACH_COLORS = {
    'Dan Campbell':  '#C5203B',
    'Andy Reid':     '#E65100',
    'Kyle Shanahan': '#1565C0',
    'Bill Belichick':'#4A148C',
    'Mike McCarthy': '#546E7A',
}

## 1. Load Data

In [ ]:
# Overall coach grades
grades = pd.read_csv(OUT_DIR + 'coach_grades.csv')
print(f'Coach grades: {len(grades)} coaches')
print(f'Columns: {grades.columns.tolist()}')

# Full graded play-by-play
df = pd.read_parquet(DATA_DIR + 'fourth_downs_graded.parquet')
print(f'\nGraded plays: {len(df):,}')

# Build coach lookup: game_id + team → coach_name
gc = pd.read_parquet(DATA_DIR + 'raw/games_coaches.parquet')
away_map = gc[['game_id', 'away_team', 'away_coach']].rename(
    columns={'away_team': 'team', 'away_coach': 'coach_name'})
home_map = gc[['game_id', 'home_team', 'home_coach']].rename(
    columns={'home_team': 'team', 'home_coach': 'coach_name'})
coach_map = pd.concat([away_map, home_map], ignore_index=True).drop_duplicates(subset=['game_id', 'team'])

# Merge coach names into graded plays
df = df.merge(
    coach_map[['game_id', 'team', 'coach_name']],
    left_on=['game_id', 'posteam'],
    right_on=['game_id', 'team'],
    how='left'
)
match_rate = df['coach_name'].notna().mean()
print(f'Coach name match rate: {match_rate*100:.1f}%')

# Verify comparison coaches are present
print('\nComparison coach check:')
for name in COMPARISON_COACHES:
    n = df[df['coach_name'] == name]['decision_gap'].notna().sum()
    in_grades = name in grades['coach_name'].values
    print(f'  {name}: {n:,} graded plays, in coach_grades={in_grades}')

## 2. Top/Bottom 15 Aggressive Coaches

In [ ]:
# Only coaches with enough decisions for reliable stats
MIN_DECISIONS = 100
qualified = grades[grades['total_decisions'] >= MIN_DECISIONS].copy()
print(f'Coaches with {MIN_DECISIONS}+ decisions: {len(qualified)}')

top15_aggressive = qualified.nlargest(15, 'go_rate')[
    ['coach_name', 'total_decisions', 'go_rate', 'dqs', 'dqs_rank', 'odr', 'seasons']
].reset_index(drop=True)

bot15_aggressive = qualified.nsmallest(15, 'go_rate')[
    ['coach_name', 'total_decisions', 'go_rate', 'dqs', 'dqs_rank', 'odr', 'seasons']
].reset_index(drop=True)

print('\nTop 15 Most Aggressive Coaches (go-for-it rate):')
print(
    top15_aggressive
    .assign(
        go_rate  = lambda x: (x['go_rate']*100).map('{:.1f}%'.format),
        dqs      = lambda x: x['dqs'].map('{:.4f}'.format),
        odr      = lambda x: (x['odr']*100).map('{:.1f}%'.format),
    )
    .to_string(index=False)
)

print('\nBottom 15 Least Aggressive Coaches:')
print(
    bot15_aggressive
    .assign(
        go_rate  = lambda x: (x['go_rate']*100).map('{:.1f}%'.format),
        dqs      = lambda x: x['dqs'].map('{:.4f}'.format),
        odr      = lambda x: (x['odr']*100).map('{:.1f}%'.format),
    )
    .to_string(index=False)
)

# Among top 15 aggressive: how many have DQS better than league median?
median_dqs = qualified['dqs'].median()
n_better = (top15_aggressive['dqs'].str.replace('%','').astype(float, errors='ignore')
            if top15_aggressive['dqs'].dtype == object
            else top15_aggressive['dqs'] < median_dqs).sum()
# recompute cleanly
top15_raw = qualified.nlargest(15, 'go_rate')
n_better = (top15_raw['dqs'] <= median_dqs).sum()
print(f'\nLeague median DQS: {median_dqs:.4f}')
print(f'Among top 15 most aggressive: {n_better}/15 have DQS at or below league median (better than avg)')

## 3. Quadrant Analysis — Archetype Classification
Divide coaches into four archetypes using median go-rate and median DQS as dividers.

In [ ]:
med_go  = qualified['go_rate'].median()
med_dqs = qualified['dqs'].median()

def archetype(row):
    aggressive = row['go_rate'] >= med_go
    accurate   = row['dqs'] <= med_dqs   # lower DQS = better = accurate
    if aggressive and accurate:     return 'Aggressive + Accurate'
    if aggressive and not accurate: return 'Aggressive + Inaccurate'
    if not aggressive and accurate: return 'Conservative + Accurate'
    return 'Conservative + Inaccurate'

qualified = qualified.copy()
qualified['archetype'] = qualified.apply(archetype, axis=1)

archetype_counts = qualified['archetype'].value_counts()
print('Archetype distribution:')
for arch, n in archetype_counts.items():
    print(f'  {arch}: {n} coaches')

print(f'\nMedian go rate: {med_go*100:.1f}%')
print(f'Median DQS:     {med_dqs:.4f}')

# Where do comparison coaches fall?
print('\nComparison coach archetypes:')
for name in COMPARISON_COACHES:
    row = qualified[qualified['coach_name'] == name]
    if len(row):
        r = row.iloc[0]
        print(f'  {name}: {r["archetype"]}  (go={r["go_rate"]*100:.1f}%, DQS={r["dqs"]:.4f}, rank #{int(r["dqs_rank"])})')
    else:
        print(f'  {name}: not in qualified set')

## 4. Season-by-Season Stats for Comparison Coaches

In [ ]:
graded_tagged = df.dropna(subset=['decision_gap', 'coach_name'])

def coach_by_season(coach_name):
    sub = graded_tagged[graded_tagged['coach_name'] == coach_name]
    if len(sub) == 0:
        return pd.DataFrame()
    return (
        sub.groupby('season')
        .agg(
            n       = ('decision_gap', 'count'),
            go_rate = ('decision',     lambda x: (x == 'go_for_it').mean()),
            dqs     = ('decision_gap', 'mean'),
            odr     = ('made_optimal', 'mean'),
        )
        .reset_index()
        .assign(coach=coach_name)
    )

all_season_stats = pd.concat(
    [coach_by_season(c) for c in COMPARISON_COACHES],
    ignore_index=True
)

print('Season-by-season graded decisions per comparison coach:')
pivot_n = all_season_stats.pivot(index='season', columns='coach', values='n').fillna(0).astype(int)
print(pivot_n.to_string())

# League average DQS by season (for reference line)
league_by_season = (
    graded_tagged
    .groupby('season')
    .agg(league_dqs = ('decision_gap', 'mean'))
    .reset_index()
)

## 5. Chart 16 — Aggression Archetype Quadrant
Every qualifying coach as a dot. Four quadrants show which archetype each coach belongs to.
Comparison coaches are annotated by name.

In [ ]:
ARCHETYPE_COLORS = {
    'Aggressive + Accurate':     '#2E7D32',
    'Aggressive + Inaccurate':   '#F57F17',
    'Conservative + Accurate':   '#1565C0',
    'Conservative + Inaccurate': '#B71C1C',
}

fig, ax = plt.subplots(figsize=(11, 8))

# Quadrant shading
ax.axhline(med_dqs, color='black', linewidth=0.8, linestyle='--', alpha=0.4, zorder=2)
ax.axvline(med_go,  color='black', linewidth=0.8, linestyle='--', alpha=0.4, zorder=2)

# Plot all coaches
for arch, color in ARCHETYPE_COLORS.items():
    sub = qualified[qualified['archetype'] == arch]
    ax.scatter(
        sub['go_rate'] * 100,
        sub['dqs'] * 100,
        color=color, alpha=0.35, s=30, zorder=3
    )

# Annotate comparison coaches
for name in COMPARISON_COACHES:
    row = qualified[qualified['coach_name'] == name]
    if len(row) == 0:
        continue
    r = row.iloc[0]
    color = COACH_COLORS.get(name, 'black')
    ax.scatter(
        r['go_rate'] * 100, r['dqs'] * 100,
        color=color, s=100, zorder=5, edgecolors='white', linewidths=1.5
    )
    offsets = {
        'Dan Campbell':   (5,  2),
        'Andy Reid':      (5,  2),
        'Kyle Shanahan':  (5, -4),
        'Bill Belichick': (5,  2),
        'Mike McCarthy':  (5,  2),
    }
    dx, dy = offsets.get(name, (5, 2))
    ax.annotate(
        name,
        xy=(r['go_rate'] * 100, r['dqs'] * 100),
        xytext=(dx, dy), textcoords='offset points',
        fontsize=7.5, color=color, fontweight='bold',
        arrowprops=dict(arrowstyle='-', color=color, lw=0.7)
    )

# Quadrant labels
x_min, x_max = ax.get_xlim()
y_min, y_max = ax.get_ylim()
label_kwargs = dict(fontsize=8, alpha=0.55, fontweight='bold', ha='center', va='center')
ax.text(med_go*100 * 0.55,  med_dqs*100 * 0.55, 'CONSERVATIVE\n+ ACCURATE',
        color=ARCHETYPE_COLORS['Conservative + Accurate'], **label_kwargs)
ax.text(med_go*100 * 1.45,  med_dqs*100 * 0.55, 'AGGRESSIVE\n+ ACCURATE',
        color=ARCHETYPE_COLORS['Aggressive + Accurate'], **label_kwargs)
ax.text(med_go*100 * 0.55,  med_dqs*100 * 1.45, 'CONSERVATIVE\n+ INACCURATE',
        color=ARCHETYPE_COLORS['Conservative + Inaccurate'], **label_kwargs)
ax.text(med_go*100 * 1.45,  med_dqs*100 * 1.45, 'AGGRESSIVE\n+ INACCURATE',
        color=ARCHETYPE_COLORS['Aggressive + Inaccurate'], **label_kwargs)

ax.set_xlabel('Go-For-It Rate (%)', fontsize=11)
ax.set_ylabel('Mean Decision Gap — DQS (lower = better)', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.invert_yaxis()   # lower DQS (better) at top
ax.set_title(
    'NFL Head Coach 4th Down Archetypes\nAggression vs Decision Quality — All Qualifying Coaches (100+ decisions)',
    fontsize=12, fontweight='bold'
)

# Legend patches
patches = [mpatches.Patch(color=c, label=a, alpha=0.7)
           for a, c in ARCHETYPE_COLORS.items()]
ax.legend(handles=patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(SAVE_DIR + '16_aggression_archetype_quadrant.png', bbox_inches='tight')
plt.show()
print('Saved: 16_aggression_archetype_quadrant.png')

## 6. Chart 17 — Top 15 Most Aggressive Coaches
Horizontal bar chart showing go-for-it rate with DQS rank as a secondary indicator.

In [ ]:
top15 = qualified.nlargest(15, 'go_rate').sort_values('go_rate').copy()
n_coaches_total = len(qualified)

# Color bars: Campbell red, others by archetype
bar_colors = [
    CAMPBELL_COLOR if name == 'Dan Campbell'
    else ARCHETYPE_COLORS.get(arch, NEUTRAL_COLOR)
    for name, arch in zip(top15['coach_name'], top15['archetype'])
]

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(
    top15['coach_name'],
    top15['go_rate'] * 100,
    color=bar_colors, edgecolor='white', linewidth=0.8, height=0.7
)

# Annotate each bar with go rate + DQS rank
for bar, (_, row) in zip(bars, top15.iterrows()):
    w = bar.get_width()
    ax.text(
        w + 0.3, bar.get_y() + bar.get_height() / 2,
        f"{w:.1f}%  (DQS rank #{int(row['dqs_rank'])}/{n_coaches_total})",
        va='center', fontsize=8, color='#37474F'
    )

# League average reference line
league_go = grades['go_rate'].mean()
ax.axvline(league_go * 100, color='black', linewidth=1, linestyle=':',
           alpha=0.5, label=f'League avg ({league_go*100:.1f}%)')

ax.set_xlabel('Go-For-It Rate (%)', fontsize=11)
ax.set_xlim(0, top15['go_rate'].max() * 100 * 1.35)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(fontsize=9)
ax.set_title(
    'Top 15 Most Aggressive NFL Coaches (100+ decisions)\nBar = Go-For-It Rate  ·  Label = DQS Rank',
    fontsize=12, fontweight='bold'
)

plt.tight_layout()
plt.savefig(SAVE_DIR + '17_top15_aggressive_coaches.png', bbox_inches='tight')
plt.show()
print('Saved: 17_top15_aggressive_coaches.png')

## 7. Chart 18 — Multi-Coach Season Trend
Season-by-season DQS for 5 comparison coaches. Campbell vs the field over time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=False)

# ── Top panel: DQS by season ──────────────────────────────────────────────────
# League average as background reference
ax1.fill_between(
    league_by_season['season'],
    league_by_season['league_dqs'] * 100,
    alpha=0.08, color='black', label='League avg (shaded)'
)
ax1.plot(
    league_by_season['season'], league_by_season['league_dqs'] * 100,
    color='black', linewidth=1, linestyle=':', alpha=0.4
)

for coach in COMPARISON_COACHES:
    sub = all_season_stats[all_season_stats['coach'] == coach]
    if len(sub) == 0:
        continue
    color = COACH_COLORS[coach]
    lw = 2.8 if coach == 'Dan Campbell' else 1.8
    ax1.plot(sub['season'], sub['dqs'] * 100, color=color,
             linewidth=lw, marker='o', markersize=5, label=coach, zorder=4)

ax1.set_ylabel('Mean DQS (lower = better)', fontsize=10)
ax1.set_title('Season-by-Season DQS — Comparison Coaches vs League Average', fontsize=11, fontweight='bold')
ax1.legend(fontsize=8, loc='upper right')
ax1.text(0.01, 0.04, 'Lower = closer to optimal decisions',
         transform=ax1.transAxes, fontsize=7.5, color='grey')
ax1.invert_yaxis()

# ── Bottom panel: go-for-it rate by season ────────────────────────────────────
# League average reference
ax2.plot(
    league_by_season['season'],
    graded_tagged.groupby('season')['decision'].apply(lambda x: (x=='go_for_it').mean()).values * 100,
    color='black', linewidth=1, linestyle=':', alpha=0.4, label='League avg'
)

for coach in COMPARISON_COACHES:
    sub = all_season_stats[all_season_stats['coach'] == coach]
    if len(sub) == 0:
        continue
    color = COACH_COLORS[coach]
    lw = 2.8 if coach == 'Dan Campbell' else 1.8
    ax2.plot(sub['season'], sub['go_rate'] * 100, color=color,
             linewidth=lw, marker='s', markersize=5, label=coach, zorder=4)

ax2.set_xlabel('Season', fontsize=10)
ax2.set_ylabel('Go-For-It Rate (%)', fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax2.set_title('Season-by-Season Go-For-It Rate', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8, loc='upper left')

# Shared x formatting
for ax in (ax1, ax2):
    ax.set_xlim(1998.5, 2025.5)
    ax.set_xticks(range(1999, 2026, 2))
    ax.tick_params(axis='x', rotation=45)

fig.suptitle(
    'NFL Coach 4th Down Comparison: 1999–2025\nDan Campbell vs Andy Reid, Shanahan, Belichick, McCarthy',
    fontsize=13, fontweight='bold', y=1.01
)

plt.tight_layout()
plt.savefig(SAVE_DIR + '18_coach_comparison_seasons.png', bbox_inches='tight')
plt.show()
print('Saved: 18_coach_comparison_seasons.png')

## 8. Save aggressive_coaches.csv

In [ ]:
aggressive_coaches = qualified[[
    'coach_name', 'total_decisions', 'go_rate', 'dqs', 'odr',
    'dqs_rank', 'odr_rank', 'seasons', 'archetype'
]].copy()
aggressive_coaches['go_rate_rank'] = aggressive_coaches['go_rate'].rank(
    ascending=False, method='min').astype(int)
aggressive_coaches = aggressive_coaches.sort_values('go_rate_rank')

aggressive_coaches.to_csv(OUT_DIR + 'aggressive_coaches.csv', index=False)

print('Saved: outputs/aggressive_coaches.csv')
print(f'Rows: {len(aggressive_coaches)}')

## 9. Summary Findings

In [ ]:
top15_raw  = qualified.nlargest(15, 'go_rate')
n_top15_accurate = (top15_raw['dqs'] <= med_dqs).sum()
dc_row     = qualified[qualified['coach_name'] == 'Dan Campbell'].iloc[0]
dc_go_rank = int(dc_row['go_rate_rank']) if 'go_rate_rank' in dc_row else None

arch_dist = qualified['archetype'].value_counts()

print('=' * 60)
print('AGGRESSIVE COACHES SUMMARY')
print('=' * 60)
print(f'Qualifying coaches (100+ decisions): {len(qualified)}')
print()
print('Archetype distribution:')
for arch, n in arch_dist.items():
    pct = n / len(qualified) * 100
    print(f'  {arch}: {n} ({pct:.0f}%)')
print()
print(f'Median go rate: {med_go*100:.1f}%')
print(f'Median DQS:     {med_dqs:.4f}')
print()
print(f'Top-15 most aggressive: {n_top15_accurate}/15 have DQS at or below median')
print(f'  → aggression {"correlates with" if n_top15_accurate >= 8 else "does NOT clearly correlate with"} better decision quality')
print()
print('Comparison coach summary:')
for name in COMPARISON_COACHES:
    row = qualified[qualified['coach_name'] == name]
    if len(row):
        r = row.iloc[0]
        print(f'  {name}: {r["archetype"]} | go={r["go_rate"]*100:.1f}% | DQS={r["dqs"]:.4f} (#{int(r["dqs_rank"])})')
print('=' * 60)